In [ ]:
from pathlib import Path

import subprocess, os


In [21]:
# Mounting dataset from D: to WSL 
WINDOWS_DRIVE = "D:"   
WINDOWS_PATH  = r"D:\HSI_Dataset\HSI_MSI_HyperResolution"  
LINK_NAME     = "data_external"  

# Mount drive if missing
mnt_path = Path(f"/mnt/{WINDOWS_DRIVE[0].lower()}")
if not mnt_path.exists():
    print(f"[INFO] Mounting {WINDOWS_DRIVE} into {mnt_path} ...")
    subprocess.run(["sudo", "mkdir", "-p", str(mnt_path)], check=True)
    res = subprocess.run(["sudo", "mount", "-t", "drvfs", WINDOWS_DRIVE, str(mnt_path)], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"Failed to mount {WINDOWS_DRIVE}: {res.stderr}")
else:
    print(f"[OK] {mnt_path} already exists")

# Verify dataset path 
dataset_path = Path(str(WINDOWS_PATH).replace("\\", "/").replace(":", "").replace("D", "/mnt/d", 1))
if not dataset_path.exists():
    print(f"[WARN] Dataset not found at {dataset_path}. Let's check what's under /mnt/d:")
    os.system("ls -la /mnt/d")
    raise FileNotFoundError("Fix dataset path above and rerun.")
print(f"[OK] Found dataset: {dataset_path}")

# Create symlink inside project
proj_root = Path.cwd()
link_path = proj_root / LINK_NAME
if link_path.exists() or link_path.is_symlink():
    print(f"[INFO] Removing old link {link_path}")
    link_path.unlink()
link_path.symlink_to(dataset_path, target_is_directory=True)
print(f"[OK] Linked {link_path} -> {dataset_path}")

# Show a few sample files for confirmation
import itertools
exts = {".hdf5", ".h5", ".hdr", ".tif", ".tiff"}
found = list(itertools.islice((p for p in link_path.rglob("*") if p.suffix.lower() in exts), 10))
if found:
    print("Sample files:")
    for f in found: print("  ", f.relative_to(link_path))
else:
    print("No .hdf5/.h5/.hdr/.tif files found yet — check deeper folders.")

[OK] /mnt/d already exists
[OK] Found dataset: /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution
[INFO] Removing old link /home/isaacmuscat/Grain-Variety-Classification/data_external
[OK] Linked /home/isaacmuscat/Grain-Variety-Classification/data_external -> /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution
Sample files:
   raw/calibration/FX10/calibration_1705a_0.hdf5
   raw/calibration/FX10/calibration_1705a_1.hdf5
   raw/calibration/FX10/calibration_1705a_2.hdf5
   raw/calibration/FX10/calibration_1705a_3.hdf5
   raw/calibration/FX10/calibration_1705_0.hdf5
   raw/calibration/FX10/calibration_1705_1.hdf5
   raw/calibration/FX10/calibration_1705_2.hdf5
   raw/calibration/FX10/calibration_1705_3.hdf5
   raw/calibration/Snapshot/processed/calibration_1705a_0/image_0000000003.hdr
   raw/calibration/Snapshot/processed/calibration_1705a_1/image_0000000000.hdr


In [ ]:
ROOT = Path("data_external")  
basepath = ROOT / "raw" / "FX10" 


In [ ]:
## Look for all files in the folder that end with .hdf5 
df = pd.DataFrame({"filepath_FX10": list(Path(f"{basepath}").rglob("**/*.hdf5"))})
## Give the sample name 
df['sample_name'] = df.filepath_FX10.apply(lambda x : x.stem)

df

,filepath_FX10,sample_name
0,data_external/raw/FX10/barley_l0.hdf5,barley_l0
1,data_external/raw/FX10/barley_l1.hdf5,barley_l1
2,data_external/raw/FX10/barley_l2.hdf5,barley_l2
3,data_external/raw/FX10/barley_l3.hdf5,barley_l3
4,data_external/raw/FX10/barley_l4.hdf5,barley_l4
...,...,...
175,data_external/raw/FX10/wheatgrass_s0.hdf5,wheatgrass_s0
176,data_external/raw/FX10/wheatgrass_s1.hdf5,wheatgrass_s1
177,data_external/raw/FX10/wheatgrass_s2.hdf5,wheatgrass_s2
178,data_external/raw/FX10/wheatgrass_s3.hdf5,wheatgrass_s3
